cleanData() is a process that are used to load all data and do standard cleaning procedure of removing duplicates and empty entries. a model cannot properly calculate empty data and duplicate entry could cause incorrect bias in the model making some movie seem more important than it is. We also change all the ids in dataset to string to ensure that they are not mistakenly used for calculations.

groupData() is created to allow simplier retrival of data need in the recommender model.
    train_users_watched create a dict of all movies that a user have watched in the training set.
    val_user_movie is used as a ground truth that we will use to run presicion checks.
    all_movies is a set of all movies that are in the training set. This is used for giving recommendations to the user

get_candidate_movies() uses train_users_watch to give a set of a movie that user have not watched. This is for only recommending movies that user have not watched before.

allData() group all the variables created for easier use when importing data into other files.


In [ ]:
import pandas as pd

def cleanData():

    train = pd.read_csv("train.csv")
    val = pd.read_csv("val.csv")

    movies = pd.read_csv(
        "movies.dat",
        sep="::",
        engine="python",
        names=["movie_id", "title", "genres"],
        encoding="latin-1"
    )

    users = pd.read_csv(
        "users.dat",
        sep="::",
        engine="python",
        names=["user_id", "gender", "age", "occupation", "zip"],
        encoding="latin-1"
    )

    train = train.drop(columns=["timestamp"]).dropna().drop_duplicates()
    val = val.drop(columns=["timestamp"]).dropna()

    train["user_id"] = train["user_id"].astype(str)
    train["movie_id"] = train["movie_id"].astype(str)

    val["user_id"] = val["user_id"].astype(str)
    val["movie_id"] = val["movie_id"].astype(str)

    movies["movie_id"] = movies["movie_id"].astype(str)
    users["user_id"] = users["user_id"].astype(str)

    return train, val, movies, users

def groupData(train, val):
    train_user_watched = train.groupby("user_id")["movie_id"].apply(set).to_dict()

    val_user_movie = val.set_index("user_id")["movie_id"].to_dict()

    all_movies = train["movie_id"].unique()

    return train_user_watched, val_user_movie, all_movies

def get_candidate_movies(user_id, train_user_watched, all_movies):
    seen = train_user_watched[user_id]
    return [m for m in all_movies if m not in seen]

def allData():
    train, val, movies, users = cleanData()

    train_user_watched, val_user_movie, all_movies = groupData(train, val)

    return {
        "train": train,
        "val": val,
        "movies": movies,
        "users": users,
        "train_user_watched": train_user_watched,
        "val_user_movie": val_user_movie,
        "all_movies": all_movies
    }

